문제 1. 계절별 평균 기온
봄(3~5월), 여름(6~8월), 가을(9~11월), 겨울(12월·1월·2월)의 도시별 평균 기온을 (5, 4) 표로 만드세요. 열 순서는 겨울, 봄, 여름, 가을입니다.
추가: 여름과 겨울 평균의 차이가 가장 큰 도시는 어디인가요?
사용할 연산: 슬라이싱, torch.cat, reshape, mean

문제 2. 전달보다 가장 많이 오른 달 · 내린 달
도시마다 기온이 전달보다 가장 많이 오른 달과 그 폭, 가장 많이 내린 달과 그 폭을 구하세요.
(예: 3월 6.0도 → 4월 12.5도이면 4월에 6.5도 오른 것)
사용할 연산: 슬라이싱, 원소별 빼기, max, argmax, argmin

문제 3. 달마다 가장 따뜻한 도시와 가장 추운 도시
1월부터 12월까지 달마다 가장 따뜻한 도시와 가장 추운 도시를 구하세요.
그리고 달마다 가장 따뜻한 도시와 가장 추운 도시의 기온 차이를 구하고, 차이가 큰 계절과 작은 계절을 말해 보세요.
사용할 연산: argmax, argmin, max, min (어느 dim을 접어야 할까요?)

문제 4. 25도 이상인 달의 수와 평균
도시마다 기온이 25도 이상인 달이 몇 개인지 구하세요.
그리고 25도 이상인 달들만의 평균 기온을 도시별로 구하세요.
사용할 연산: 비교 마스크, sum, torch.where

문제 5. 행렬 곱으로 분기 평균 구하기
1분기(1~3월), 2분기(4~6월), 3분기(7~9월), 4분기(10~12월)의 도시별 평균을 행렬 곱 한 번으로 구하세요.
(12, 4) 크기의 행렬을 직접 만들어 기온 @ 행렬 로 계산하고, reshape와 mean으로 구한 결과와 같은지 torch.allclose로 확인하세요.
사용할 연산: torch.zeros, 슬라이싱 대입, @, reshape, mean, allclose


In [5]:
# [2권 Chapter 4. 텐서: 숫자를 담는 그릇 · 4.11 실습: 도시별 기후 패턴 비교하기 · manuscript-vol2/chapters/04-tensors.md:1334]
import torch

도시 = ["가람시", "나래시", "다솜시", "라온시", "마루시"]
# 도시 5곳 × 12개월 월평균 기온(°C). 실제 관측값이 아닌 예제용 숫자입니다.
기온 = torch.tensor(
    [
        [-2.0, 0.5, 6.0, 12.5, 18.0, 22.5, 25.5, 26.0, 21.5, 14.5, 7.0, 0.5],
        [3.5, 5.0, 9.0, 14.0, 18.5, 21.5, 25.0, 26.5, 23.0, 18.0, 12.0, 6.0],
        [-4.0, -1.5, 4.5, 11.5, 17.5, 22.0, 24.5, 25.0, 19.5, 12.5, 5.0, -1.5],
        [6.0, 7.0, 10.0, 14.5, 18.5, 21.5, 26.0, 27.0, 23.5, 19.0, 13.5, 8.5],
        [-1.0, 1.0, 6.5, 13.0, 18.5, 23.0, 25.5, 26.5, 21.0, 14.0, 7.5, 1.0],
    ]
)
# print("모양:", 기온.shape)

# 1) 도시별 연평균과 연교차(가장 더운 달 - 가장 추운 달)
연평균 = 기온.mean(dim=1)
연교차 = 기온.max(dim=1).values - 기온.min(dim=1).values
for 이름, 평균, 교차 in zip(도시, 연평균, 연교차):
    # print(f"{이름}: 연평균 {평균:5.2f}°C, 연교차 {교차:4.1f}°C")
    pass

# 2) 도시마다 가장 더운 달 (argmax는 0부터 세므로 +1)
# print("가장 더운 달:", (기온.argmax(dim=1) + 1).tolist())

print()
# print("가장 추운 달:", (기온.argmin(dim=1) + 1).tolist())
# print(기온.min(dim=1))
# print("도시별 영하인 달의 갯수:", (기온 < 0).sum(dim=1))
# print("연교차가 가장 큰 도시:", 도시[연교차.argmax()])
print()

# 3) 도시마다 정규화: (기온 - 그 도시 평균) / 그 도시 표준편차
평균 = 기온.mean(dim=1, keepdim=True)  # (5, 1)
표준편차 = 기온.std(dim=1, keepdim=True)  # (5, 1)
정규화 = (기온 - 평균) / 표준편차  # (5, 12) - (5, 1) → 브로드캐스팅
# print("정규화 후 도시별 평균:", 정규화.mean(dim=1).abs().max() < 1e-6)

# 월 별 다섯 도시의 평균 기온과 shape
# print("월 별 다섯 도시의 평균 기온:", 기온.mean(dim=0))
# print("월 별 다섯 도시의 평균 기온 (shape):", 기온.mean(dim=0).shape)

# 4) 기후 패턴이 가장 닮은 도시 찾기: 코사인 유사도 = 단위 벡터끼리의 행렬곱
단위 = 정규화 / 정규화.norm(dim=1, keepdim=True)  # keepdim=True로 차원 유지
유사도 = 단위 @ 단위.T  # (5, 12) @ (12, 5) → (5, 5)
유사도.fill_diagonal_(-1.0)  # 자기 자신은 빼기
짝 = 유사도.argmax(dim=1)
for i, j in enumerate(짝.tolist()):
    # print(f"{도시[i]}와 가장 닮은 도시: {도시[j]} (유사도 {유사도[i, j]:.4f})")
    pass

# 5) 도시마다 가장 더운 석 달 (topk는 큰 순서로 주므로 달 순서로 다시 정렬)
더운_석달 = 기온.topk(3, dim=1).indices.sort(dim=1).values + 1
for 이름, 달 in zip(도시, 더운_석달.tolist()):
    # print(f"{이름}의 가장 더운 석 달: {달}월")
    pass

# 6) 난방 지수: 10도보다 추운 만큼을 달마다 더한 값 (예제용 지표)
난방 = torch.where(기온 < 10, 10 - 기온, 0.0)
# print("난방이 필요한 달 수:", (기온 < 10).sum(dim=1).tolist())
# print("난방 지수:", 난방.sum(dim=1).tolist())
# print("clamp로 계산해도 같은가:", torch.equal(난방, (10 - 기온).clamp(min=0)))

# warming_standard to 15
warming_standard = torch.where(기온 < 15, 15 - 기온, 0.0)
# print("난방이 필요한 달 수 (15도 기준):", (기온 < 15).sum(dim=1).tolist())
# print("난방 지수 (15도 기준):", warming_standard.sum(dim=1).tolist())

###
# 문제 1. 계절별 평균 기온
# 봄(3~5월), 여름(6~8월), 가을(9~11월), 겨울(12월·1월·2월)의 도시별 평균 기온을 (5, 4) 표로 만드세요. 열 순서는 겨울, 봄, 여름, 가을입니다.
# 추가: 여름과 겨울 평균의 차이가 가장 큰 도시는 어디인가요?
# 사용할 연산: 슬라이싱, torch.cat, reshape, mean
spring_summer_fall = 기온[:, 2:11].reshape(5, 3, 3).mean(dim=2)  # (5, 3) = 봄, 여름, 가을
winter = torch.cat([기온[:, 11:], 기온[:, :2]], dim=1)  # 12월, 1월, 2월 합치기
winter = winter.mean(dim=1, keepdim=True)  # (5, 1) = 겨울 평균
season = torch.cat([winter, spring_summer_fall], dim=1)  # (5, 4) = 겨울, 봄, 여름, 가을
diff = season[:, 2] - season[:, 0]  # 여름 - 겨울
print("계절별 평균 기온 (열 순서: 겨울, 봄, 여름, 가을):")
print(season)
print("여름과 겨울 평균의 차이가 가장 큰 도시:", 도시[diff.argmax()])

# 문제 2. 전달보다 가장 많이 오른 달 · 내린 달
# 도시마다 기온이 전달보다 가장 많이 오른 달과 그 폭, 가장 많이 내린 달과 그 폭을 구하세요.
# (예: 3월 6.0도 → 4월 12.5도이면 4월에 6.5도 오른 것)
# 사용할 연산: 슬라이싱, 원소별 빼기, max, argmax, argmin
changes = 기온[:, 1:] - 기온[:, :-1]  # 전달 대비 변화량  # (5, 11) = 1월~11월 대비 2월~12월 변화량
print("가장 많이 오른 달과 그 폭:")
print(changes.argmax(dim=1) + 2, changes.max(dim=1).values)  # +2: 변화가 일어난 달 (2월~12월)
print("가장 많이 내린 달과 그 폭:")
print(changes.argmin(dim=1) + 2, changes.min(dim=1).values)  # +2: 변화가 일어난 달 (2월~12월)



계절별 평균 기온 (열 순서: 겨울, 봄, 여름, 가을):
tensor([[-0.3333, 12.1667, 24.6667, 14.3333],
        [ 4.8333, 13.8333, 24.3333, 17.6667],
        [-2.3333, 11.1667, 23.8333, 12.3333],
        [ 7.1667, 14.3333, 24.8333, 18.6667],
        [ 0.3333, 12.6667, 25.0000, 14.1667]])
여름과 겨울 평균의 차이가 가장 큰 도시: 다솜시
가장 많이 오른 달과 그 폭:
tensor([4, 4, 4, 4, 4]) tensor([6.5000, 5.0000, 7.0000, 4.5000, 6.5000])
가장 많이 내린 달과 그 폭:
tensor([11, 11, 11, 11, 10]) tensor([-7.5000, -6.0000, -7.5000, -5.5000, -7.0000])


In [2]:
import torch


def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")  # NVIDIA GPU
    if torch.backends.mps.is_available():
        return torch.device("mps")  # Apple 실리콘 GPU
    return torch.device("cpu")


device = get_device()
print(device)


cuda


# [Tensors](https://docs.pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html#tensors)


In [3]:
import numpy as np


data = [[1, 2], [3, 4]]
x_data = torch.tensor(data)

x_data

tensor([[1, 2],
        [3, 4]])

In [5]:
np_array = np.array(data)
x_np = torch.from_numpy(np_array)

x_np

tensor([[1, 2],
        [3, 4]])

In [7]:
x_ones = torch.ones_like(x_data)
print(f"Ones Tensor: \n {x_ones} \n")

x_rand = torch.rand_like(x_data, dtype=torch.float)
print(f"Random Tensor: \n {x_rand} \n")

Ones Tensor: 
 tensor([[1, 1],
        [1, 1]]) 

Random Tensor: 
 tensor([[0.4535, 0.5972],
        [0.2704, 0.8383]]) 



In [8]:
shape = (2, 3)
rand_tensor = torch.rand(shape)
ones_tensor = torch.ones(shape)
zeros_tensor = torch.zeros(shape)

print(f"Random Tensor: \n {rand_tensor} \n")
print(f"Ones Tensor: \n {ones_tensor} \n")
print(f"Zeros Tensor: \n {zeros_tensor} \n")

Random Tensor: 
 tensor([[0.7035, 0.5078, 0.0486],
        [0.5980, 0.7695, 0.8565]]) 

Ones Tensor: 
 tensor([[1., 1., 1.],
        [1., 1., 1.]]) 

Zeros Tensor: 
 tensor([[0., 0., 0.],
        [0., 0., 0.]]) 



In [9]:
tensor = torch.rand(3, 4)

print(f"Shape of tensor: {tensor.shape}")
print(f"Datatype of tensor: {tensor.dtype}")
print(f"Device tensor is stored on: {tensor.device}")

Shape of tensor: torch.Size([3, 4])
Datatype of tensor: torch.float32
Device tensor is stored on: cpu


In [10]:
if torch.accelerator.is_available():
    tensor = tensor.to(torch.accelerator.current_accelerator())


print(f"Tensor is now on device: {tensor.device}")

Tensor is now on device: cuda:0


In [13]:
tensor = torch.ones(4, 4)
print(f"First row: {tensor[0]}")
print(f"First column: {tensor[:, 0]}")
print(f"Last column: {tensor[:, -1]}")
print(f"Last column: {tensor[..., -1]}")
print()
tensor[:, 1] = 0
print(f"Tensor after setting second column to 0: \n{tensor}")

First row: tensor([1., 1., 1., 1.])
First column: tensor([1., 1., 1., 1.])
Last column: tensor([1., 1., 1., 1.])
Last column: tensor([1., 1., 1., 1.])

Tensor after setting second column to 0: 
tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]])


In [15]:
print(tensor)
print()

print(torch.cat([tensor, tensor, tensor], dim=1))  # row-wise concatenation
print()
print(torch.cat([tensor, tensor, tensor], dim=0))  # column-wise concatenation

tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]])

tensor([[1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1.],
        [1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1.],
        [1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1.],
        [1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1.]])

tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]])


In [21]:
print(f"tensor:\n {tensor}")

# This computes the matrix multiplication between two tensors. y1, y2, y3 will have the same value
# ``tensor.T`` returns the transpose of a tensor
y1 = tensor @ tensor.T
y2 = tensor.matmul(tensor.T)
print(f"y1:\n {y1}")
print(f"y2:\n {y2}")
print(f"y1 equal y2: {torch.allclose(y1, y2)}")
print()

y3 = torch.rand_like(y1)
torch.matmul(tensor, tensor.T, out=y3)
print(f"y3:\n {y3}")
print(f"y3 equal y1: {torch.allclose(y3, y1)}")
print()

# This computes the element-wise product. z1, z2, z3 will have the same value
z1 = tensor * tensor
z2 = tensor.mul(tensor)

z3 = torch.rand_like(tensor)
torch.mul(tensor, tensor, out=z3)

tensor:
 tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]])
y1:
 tensor([[3., 3., 3., 3.],
        [3., 3., 3., 3.],
        [3., 3., 3., 3.],
        [3., 3., 3., 3.]])
y2:
 tensor([[3., 3., 3., 3.],
        [3., 3., 3., 3.],
        [3., 3., 3., 3.],
        [3., 3., 3., 3.]])
y1 equal y2: True

y3:
 tensor([[3., 3., 3., 3.],
        [3., 3., 3., 3.],
        [3., 3., 3., 3.],
        [3., 3., 3., 3.]])
y3 equal y1: True



tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]])

이 코드의 핵심은 **`@`와 `matmul()`은 행렬곱**, **`*`와 `mul()`은 원소별 곱**이라는 차이야. 겉보기엔 둘 다 “곱셈”이지만 계산 규칙이 완전히 달라.

### 텐서를 행벡터로 보기

```text
tensor =
[1, 0, 1, 1]
[1, 0, 1, 1]
[1, 0, 1, 1]
[1, 0, 1, 1]
```

모양은 `(4, 4)`야. 네 행이 전부 똑같은 벡터 `[1, 0, 1, 1]`이지.

### 1) `tensor @ tensor.T`: 행렬곱

```python
y1 = tensor @ tensor.T
```

`tensor.T`는 전치야. 행과 열을 바꿔. 여기서는 모든 행이 똑같아서 전치해도 모양과 값이 똑같아 보여.

행렬곱의 각 결과 원소는 **왼쪽 텐서의 한 행과 오른쪽 텐서의 한 열을 곱해서 더한 값**이야. 예를 들어 결과의 첫 번째 원소는:

\[
1\times1 + 0\times0 + 1\times1 + 1\times1 = 3
\]

모든 행과 열이 똑같은 `[1, 0, 1, 1]`이므로, 결과의 모든 원소가 3이 돼.

```text
y1 =
[3, 3, 3, 3]
[3, 3, 3, 3]
[3, 3, 3, 3]
[3, 3, 3, 3]
```

모양도 `(4, 4)`야. 일반 규칙은 `(m, n) @ (n, p) → (m, p)`이고, 여기서는 `(4, 4) @ (4, 4) → (4, 4)`야.

```python
y2 = tensor.matmul(tensor.T)
```

이건 같은 행렬곱을 메서드 형태로 쓴 거야. 그래서 `y1`과 `y2`의 원소가 같아. `@` 연산자, `.matmul()`, `torch.matmul()`은 행렬곱을 표현하는 서로 다른 표기법이야. <citation src="2"></citation>

```python
torch.allclose(y1, y2)
```

두 텐서의 값이 허용 오차 안에서 가까운지 확인해. `True`라는 건 `y1`과 `y2`의 원소들이 서로 같다고 볼 만큼 가깝다는 뜻이야. 부동소수점 연산은 장치나 계산 순서에 따라 미세한 오차가 생길 수 있어서, 단순히 완전히 같은지 비교하는 대신 `allclose`를 자주 사용해.

### 2) `out=y3`: 결과를 지정한 텐서에 쓰기

```python
y3 = torch.rand_like(y1)
torch.matmul(tensor, tensor.T, out=y3)
```

`torch.rand_like(y1)`은 `y1`과 같은 모양·자료형·장치를 가진 텐서를 만들고, 일단 난수로 채워. 그런데 바로 다음 줄에서 `torch.matmul(..., out=y3)`가 행렬곱 결과를 `y3`에 써 넣으므로, 처음 난수 값은 덮어써져.

따라서 마지막에 출력되는 `y3`도 전부 3이야. 이때 “`y1`, `y2`, `y3`가 같은 값”이라는 말은 **원소 값이 같다**는 뜻이지, 세 변수가 같은 텐서 객체나 같은 메모리를 공유한다는 뜻은 아니야.

`out=`은 결과를 어디에 기록할지 지정하는 방식이야. 미리 결과 저장 공간을 준비하는 저수준·메모리 관리 용도로 볼 수 있어. 보통의 코드에서는 `y3 = torch.matmul(...)`처럼 쓰는 편이 읽기 쉽고, `out=` 연산에는 자동 미분 관련 제약이 있을 수 있으니 gradient를 추적해야 하는 학습 코드에서는 특히 주의해야 해.

### 3) `tensor * tensor`: 원소별 곱

```python
z1 = tensor * tensor
```

이건 행렬곱이 아니라, **같은 위치의 원소끼리 곱하는 연산**이야.

```text
[1, 0, 1, 1] * [1, 0, 1, 1]
→ [1×1, 0×0, 1×1, 1×1]
→ [1, 0, 1, 1]
```

전체 텐서에서도 같은 위치끼리 계산하므로 결과는 원래 텐서와 똑같이 생겨. `z2 = tensor.mul(tensor)`와 `z3 = torch.mul(tensor, tensor, out=z3)`도 이 원소별 곱을 각각 메서드 형태, `out=` 형태로 표현한 거야. PyTorch 튜토리얼도 `@`/`matmul`과 `*`/`mul`을 이처럼 구분해 설명해. <citation src="2"></citation>

한 가지 눈여겨볼 점: 이 코드 조각에서는 `z1`, `z2`, `z3`를 계산만 하고 **출력하는 `print` 문은 없어**. 따라서 화면에 보이는 출력에는 행렬곱 결과만 나오고, 원소별 곱 결과는 표시되지 않아.

요약하면:

- `A @ B`: 행·열을 조합해 곱하고 더하는 **행렬곱**
- `A * B`: 같은 위치끼리 곱하는 **원소별 곱**
- `.matmul()` / `torch.matmul()`: 행렬곱 표현
- `.mul()` / `torch.mul()`: 원소별 곱 표현
- `out=어떤_텐서`: 결과를 그 텐서에 기록하는 방식

In [22]:
agg = tensor.sum()
agg_item = agg.item()
print(f"agg: {agg}")
print(f"agg_item: {agg_item}, type: {type(agg_item)}")


agg: 12.0
agg_item: 12.0, type: <class 'float'>


In [ ]:
print(f"{tensor} \n")
print(f"tensor shape: {tensor.shape}")
print(f"tensor dtype: {tensor.dtype}")

# In-place operations Operations that store the result into the operand are called in-place. They are denoted by a _ suffix. For example: x.copy_(y), x.t_(), will change x.
tensor.add_(5)
print(f"tensor after add_:\n {tensor}")

# In-place operations save some memory, but can be problematic when computing derivatives because of an immediate loss of history. Hence, their use is discouraged.


tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]]) 

tensor shape: torch.Size([4, 4])
tensor dtype: torch.float32
tensor after add_:
 tensor([[6., 5., 6., 6.],
        [6., 5., 6., 6.],
        [6., 5., 6., 6.],
        [6., 5., 6., 6.]])


# [Bridge with NumPy](https://docs.pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html#bridge-with-numpy)

Tensors on the CPU and NumPy arrays can share their underlying memory locations, and changing one will change the other.

In [25]:
t = torch.ones(5)
print(f"t:\n {t}")
print(f"t shape: {t.shape}")
print(f"t dtype: {t.dtype}")

print()

n = t.numpy()
print(f"n:\n {n}")
print(f"n shape: {n.shape}")
print(f"n dtype: {n.dtype}")

t:
 tensor([1., 1., 1., 1., 1.])
t shape: torch.Size([5])
t dtype: torch.float32

n:
 [1. 1. 1. 1. 1.]
n shape: (5,)
n dtype: float32


In [28]:
t.add_(1)
print(f"t after add_:\n {t}")
print(f"t shape: {t.shape}")
print(f"t dtype: {t.dtype}")

print()

print(f"n after t.add_:\n {n}")
print(f"n shape: {n.shape}")
print(f"n dtype: {n.dtype}")

# The numpy array n is also updated because it shares the same memory with the tensor t.

t after add_:
 tensor([4., 4., 4., 4., 4.])
t shape: torch.Size([5])
t dtype: torch.float32

n after t.add_:
 [4. 4. 4. 4. 4.]
n shape: (5,)
n dtype: float32


# [NumPy array to Tensor](https://docs.pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html#numpy-array-to-tensor)

In [29]:
n = np.ones(5)
t = torch.from_numpy(n)

print(f"n:\n {n}")
print(f"n shape: {n.shape}")
print(f"n dtype: {n.dtype}")

print()

print(f"t:\n {t}")
print(f"t shape: {t.shape}")
print(f"t dtype: {t.dtype}")



n:
 [1. 1. 1. 1. 1.]
n shape: (5,)
n dtype: float64

t:
 tensor([1., 1., 1., 1., 1.], dtype=torch.float64)
t shape: torch.Size([5])
t dtype: torch.float64


In [30]:
np.add(n, 1, out=n)
print(f"n after np.add:\n {n}")
print(f"n shape: {n.shape}")
print(f"n dtype: {n.dtype}")
print()
print(f"t after np.add:\n {t}")
print(f"t shape: {t.shape}")
print(f"t dtype: {t.dtype}")

# The tensor t is also updated because it shares the same memory with the numpy array n.

n after np.add:
 [2. 2. 2. 2. 2.]
n shape: (5,)
n dtype: float64

t after np.add:
 tensor([2., 2., 2., 2., 2.], dtype=torch.float64)
t shape: torch.Size([5])
t dtype: torch.float64


In [ ]:
# https://docs.pytorch.org/docs/2.15/generated/torch.stack.html

---

# [Introduction to PyTorch Tensors](https://docs.pytorch.org/tutorials/beginner/introyt/tensors_deeper_tutorial.html)

In [ ]:
import math
import torch


In [32]:
x = torch.empty(3, 4)
print(f"x:\n {x}")
print(f"type(x): {type(x)}")
print(f"x shape: {x.shape}")
print(f"x dtype: {x.dtype}")

x:
 tensor([[ 2.0000e+00,  1.8009e+00, -1.0842e-19,  1.7519e+00],
        [ 0.0000e+00,  1.3195e+00,  0.0000e+00,  1.7745e+00],
        [-0.0000e+00,  1.8174e+00, -0.0000e+00,  1.8391e+00]])
type(x): <class 'torch.Tensor'>
x shape: torch.Size([3, 4])
x dtype: torch.float32


In [33]:
print(torch.zeros(2, 3))
print(torch.ones(2, 3))

torch.manual_seed(1729)
print(torch.rand(2, 3))

tensor([[0., 0., 0.],
        [0., 0., 0.]])
tensor([[1., 1., 1.],
        [1., 1., 1.]])
tensor([[0.3126, 0.3791, 0.3087],
        [0.0736, 0.4216, 0.0691]])


In [ ]:
torch.manual_seed(1729)
random1 = torch.rand(2, 3)
print(random1)

random2 = torch.rand(2, 3)
print(random2)

print()

torch.manual_seed(1729)
random3 = torch.rand(2, 3)
print(random3)

random4 = torch.rand(2, 3)
print(random4)

# What you should see above is that random1 and random3 carry identical values, as do random2 and random4. Manually setting the RNG’s seed resets it, so that identical computations depending on random number should, in most settings, provide identical results.


tensor([[0.3126, 0.3791, 0.3087],
        [0.0736, 0.4216, 0.0691]])
tensor([[0.2332, 0.4047, 0.2162],
        [0.9927, 0.4128, 0.5938]])

tensor([[0.3126, 0.3791, 0.3087],
        [0.0736, 0.4216, 0.0691]])
tensor([[0.2332, 0.4047, 0.2162],
        [0.9927, 0.4128, 0.5938]])


In [36]:
x = torch.empty(2, 2, 3)
print(x.shape)
print(x)

empty_like_x = torch.empty_like(x)
print(empty_like_x.shape)
print(empty_like_x)

zeros_like_x = torch.zeros_like(x)
print(zeros_like_x.shape)
print(zeros_like_x)

ones_like_x = torch.ones_like(x)
print(ones_like_x.shape)
print(ones_like_x)

rand_like_x = torch.rand_like(x)
print(rand_like_x.shape)
print(rand_like_x)

torch.Size([2, 2, 3])
tensor([[[1., 1., 1.],
         [1., 1., 1.]],

        [[1., 1., 1.],
         [1., 1., 1.]]])
torch.Size([2, 2, 3])
tensor([[[1., 1., 1.],
         [1., 1., 1.]],

        [[1., 1., 1.],
         [1., 1., 1.]]])
torch.Size([2, 2, 3])
tensor([[[0., 0., 0.],
         [0., 0., 0.]],

        [[0., 0., 0.],
         [0., 0., 0.]]])
torch.Size([2, 2, 3])
tensor([[[1., 1., 1.],
         [1., 1., 1.]],

        [[1., 1., 1.],
         [1., 1., 1.]]])
torch.Size([2, 2, 3])
tensor([[[0.6128, 0.1519, 0.0453],
         [0.5035, 0.9978, 0.3884]],

        [[0.6929, 0.1703, 0.1384],
         [0.4759, 0.7481, 0.0361]]])


In [37]:
print(torch.tensor([[3.1415926, 2.71828], [1.61803, 0.0072897]]))
print(torch.tensor((2, 3, 5, 7, 11, 13, 17, 19)))
print(torch.tensor((
    (2, 4, 6),
    [3, 6, 9]
)))

tensor([[3.1416, 2.7183],
        [1.6180, 0.0073]])
tensor([ 2,  3,  5,  7, 11, 13, 17, 19])
tensor([[2, 4, 6],
        [3, 6, 9]])


In [38]:
print(torch.ones((2, 3), dtype=torch.int16))

b = torch.rand((2, 3), dtype=torch.float64) * 20.0
print(b)

print(b.to(torch.int32))

tensor([[1, 1, 1],
        [1, 1, 1]], dtype=torch.int16)
tensor([[ 0.9956,  1.4148,  5.8364],
        [11.2406, 11.2083, 11.6692]], dtype=torch.float64)
tensor([[ 0,  1,  5],
        [11, 11, 11]], dtype=torch.int32)


In [39]:
ones = torch.zeros(2, 2) + 1
twos = torch.ones(2, 2) * 2
threes = (torch.ones(2, 2) * 7 - 1) / 2
fours = twos ** 2
sqrt2s = twos ** 0.5

print(ones)
print(twos)
print(threes)
print(fours)
print(sqrt2s)

tensor([[1., 1.],
        [1., 1.]])
tensor([[2., 2.],
        [2., 2.]])
tensor([[3., 3.],
        [3., 3.]])
tensor([[4., 4.],
        [4., 4.]])
tensor([[1.4142, 1.4142],
        [1.4142, 1.4142]])


In [40]:
print(twos ** torch.tensor([
    [1, 2],
    [3, 4]
]))

print(ones + fours)

print(threes * fours)


tensor([[ 2.,  4.],
        [ 8., 16.]])
tensor([[5., 5.],
        [5., 5.]])
tensor([[12., 12.],
        [12., 12.]])


In [42]:
# It’s important to note here that all of the tensors in the previous code cell were of identical shape. What happens when we try to perform a binary operation on tensors if dissimilar shape?

# The following cell throws a run-time error. This is intentional.
try:
    print(torch.rand((2, 3) * torch.rand((3, 2))))
except TypeError as e:
    print(f"TypeError: {e}")

# In the general case, you cannot operate on tensors of different shape this way, even in a case like the cell above, where the tensors have an identical number of elements.

# To handle such cases, PyTorch provides broadcasting, which allows certain operations on tensors of different shapes by automatically expanding their dimensions to be compatible.


TypeError: only integer tensors of a single element can be converted to an index


In [43]:
# The exception to the same-shapes rule is tensor broadcasting. Here’s an example:

rand = torch.rand(2, 4)
doubled = rand * (torch.ones(1, 4) * 2)

print(rand)
print(doubled)

# Here, the tensor `torch.ones(1, 4) * 2` is broadcasted to match the shape of `rand` before the multiplication.
# Broadcasting works by expanding the smaller tensor along the dimensions where it has size 1, so that it matches the shape of the larger tensor.
# In this example, `torch.ones(1, 4) * 2` has shape `(1, 4)`, and `rand` has shape `(2, 4)`. The smaller tensor is broadcasted along the first dimension to match the shape of `rand`.
# This allows the element-wise multiplication to be performed without any shape mismatch errors.

tensor([[0.2024, 0.5731, 0.7191, 0.4067],
        [0.7301, 0.6276, 0.7357, 0.0381]])
tensor([[0.4049, 1.1461, 1.4382, 0.8134],
        [1.4602, 1.2551, 1.4715, 0.0762]])


In [48]:
# The rules for broadcasting are:
#     Each tensor must have at least one dimension - no empty tensors.
#     Comparing the dimension sizes of the two tensors, going from last to first:
#         Each dimension must be equal, or
#         One of the dimensions must be of size 1, or
#         The dimension does not exist in one of the tensors
# Tensors of identical shape, of course, are trivially “broadcastable”, as you saw earlier.
# Here are some examples of situations that honor the above rules and allow broadcasting:

a = torch.ones(4, 3, 2)
print(a)

print(a * torch.rand(3, 2))  # 3rd & 2nd dims identical to a, dim 1 absent
print(a * torch.rand(3, 1))  # 3rd dim = 1, 2nd dim identical to a
print(a * torch.rand(1, 2))  # 3rd dim identical to a, 2nd dim = 1


# The following cell throws a run-time error. This is intentional.
try:
    a * torch.rand(4, 3)  # dimensions must match last-to-first
    a * torch.rand(2, 3)  # both 3rd & 2nd dims different
    a * torch.rand((0,))  # can't broadcast with an empty tensor
except RuntimeError as e:
    print(e)  # RuntimeError: The size of tensor a (2) must match the size of tensor b (3) at non-singleton dimension 2


tensor([[[1., 1.],
         [1., 1.],
         [1., 1.]],

        [[1., 1.],
         [1., 1.],
         [1., 1.]],

        [[1., 1.],
         [1., 1.],
         [1., 1.]],

        [[1., 1.],
         [1., 1.],
         [1., 1.]]])
tensor([[[0.4408, 0.3877],
         [0.6078, 0.2345],
         [0.7117, 0.7402]],

        [[0.4408, 0.3877],
         [0.6078, 0.2345],
         [0.7117, 0.7402]],

        [[0.4408, 0.3877],
         [0.6078, 0.2345],
         [0.7117, 0.7402]],

        [[0.4408, 0.3877],
         [0.6078, 0.2345],
         [0.7117, 0.7402]]])
tensor([[[0.4467, 0.4467],
         [0.8499, 0.8499],
         [0.1399, 0.1399]],

        [[0.4467, 0.4467],
         [0.8499, 0.8499],
         [0.1399, 0.1399]],

        [[0.4467, 0.4467],
         [0.8499, 0.8499],
         [0.1399, 0.1399]],

        [[0.4467, 0.4467],
         [0.8499, 0.8499],
         [0.1399, 0.1399]]])
tensor([[[0.9280, 0.1986],
         [0.9280, 0.1986],
         [0.9280, 0.1986]],

        [[0.9280,

# [More Math with Tensors](https://docs.pytorch.org/tutorials/beginner/introyt/tensors_deeper_tutorial.html#more-math-with-tensors)

In [49]:
# common functions
a = torch.rand(2, 4) * 2 - 1
print("common functions")
print(a)
print(torch.abs(a))
print(torch.ceil(a))
print(torch.floor(a))
print(torch.clamp(a, -0.5, 0.5))

common functions
tensor([[ 0.4975,  0.8212,  0.7883,  0.5985],
        [-0.8892,  0.1552,  0.8260,  0.9665]])
tensor([[0.4975, 0.8212, 0.7883, 0.5985],
        [0.8892, 0.1552, 0.8260, 0.9665]])
tensor([[1., 1., 1., 1.],
        [-0., 1., 1., 1.]])
tensor([[ 0.,  0.,  0.,  0.],
        [-1.,  0.,  0.,  0.]])
tensor([[ 0.4975,  0.5000,  0.5000,  0.5000],
        [-0.5000,  0.1552,  0.5000,  0.5000]])


In [50]:
# trigonometric functions and their inverses
angles = torch.tensor([0, math.pi / 4, math.pi / 2, 3 * math.pi / 4])
sines = torch.sin(angles)
inverses = torch.asin(sines)
print("\nSine and arcsine:")
print("angles:", angles)
print("sines:", sines)
print("inverses:", inverses)


Sine and arcsine:
angles: tensor([0.0000, 0.7854, 1.5708, 2.3562])
sines: tensor([0.0000, 0.7071, 1.0000, 0.7071])
inverses: tensor([0.0000, 0.7854, 1.5708, 0.7854])


In [51]:
# bitwise operations
print("\nBitwise XOR:")
b = torch.tensor([1, 5, 11])
c = torch.tensor([2, 7, 10])
print(torch.bitwise_xor(b, c))


Bitwise XOR:
tensor([3, 2, 1])
